# Introduction

Using the Kaggle house price prediction competition to learn about linear regression.

* This notebook was created using an iterative approach, checking what works and what doesn't by trial and error. Many changes had only minor influence on the competition score.
* The notebook uses a functional approach to build a **reproducible feature engineering pipeline** that is first applied to the training data while building the model and later on to the test data to create the submission for the competition.
* Changes between the versions including the resulting competition score are documented in the changelog section at the end of the notebook.
* The model build in this notebook consistently reaches a competition score of around 0.138 which is better than the median score of 0.145, see ["House Prices What is a Good Score?"](https://www.kaggle.com/code/fedesoriano/house-prices-what-s-a-good-score) but not great.
* The best competition score achieved by this notebook was for version 13 with 0.136. Attempts to improve the model by feature selection and introduction of some 2nd order polynominals made the score slightly worse.

## Todo

- Improve feature engineering, e.g., can we build a model predicting price per area?
- Some notebooks such as [this one](https://www.kaggle.com/code/apapiu/regularized-linear-models/notebook) and [this](https://www.kaggle.com/code/juliencs/a-study-on-regression-applied-to-the-ames-dataset) use log transforms for features. Their residual plots look better than mine. See also [stackexchange discussion on when and how to use log transformations](https://stats.stackexchange.com/questions/18844/when-and-why-should-you-take-the-log-of-a-distribution-of-numbers).
- Better way to choose the ragularization parameter for the regression, see [this example](https://www.kaggle.com/code/apapiu/regularized-linear-models/notebook)

In [ ]:
from functools import reduce, partial

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
print("The pandas version is {}.".format(pd.__version__))

import sklearn
print('The scikit-learn version is {}.'.format(sklearn.__version__))
#sklearn.set_config(transform_output="pandas")  # requires sklearn version >= 1.2
from sklearn import feature_selection, linear_model, metrics, model_selection, pipeline, preprocessing
import category_encoders as ce

import scipy.stats as stats
import statsmodels.api as sm

# Seaborn
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Exploratory data analysis

## Load data and first look

In [ ]:
def load_data(filename):
    """Reads data set from file.
    
    Args:
    filename (str)
    
    Returns:
    pandas.DataFrame
    """
    data_dir = "/kaggle/input/house-prices-advanced-regression-techniques"
    na_value_list = ["", "N/A", "n/a", "nan"]  # NA is valid data in this data set, see check for missing values below
    train_data_file = os.path.join(data_dir, filename)
    df = pd.read_csv(train_data_file, na_values=na_value_list, keep_default_na=False)
    return df

df_train = load_data("train.csv")

In [ ]:
df_train.sample(5)

## First look at sale prices

Goal is to predict sale prices, so let's take a closer look at the sales prices in the training data set. They are in the column "SalePrice".

In [ ]:
df_train["SalePrice"].describe()

In [ ]:
print("Target skewness: {}".format(df_train["SalePrice"].skew()))
print("target kurtosis: {}".format(df_train["SalePrice"].kurtosis()))

sns.displot(data=df_train, x="SalePrice")

The target is not normally distributed. In many notebooks published on Kaggle log transforms are used. On one hand, linear regression does not assume that the target is normally distributed. One the other hand, houses with higher prices most likely also have higher variance and *constant variance* is one of the [assumptions for linear regression](https://en.wikipedia.org/wiki/Linear_regression#Assumptions). Using a log transformation of the target helps to reduce the variance for higher target values. Furthermore, the competition scoring is also using log scaled values.

We will use a log transform if ```LOG_TRANSFORM_TARGET``` is True. After regression, the transformation is reversed.

In [ ]:
LOG_TRANSFORM_TARGET = True

if LOG_TRANSFORM_TARGET:
    df_train["SalePrice"] = np.log(df_train["SalePrice"])

In [ ]:
print("Target skewness: {}".format(df_train["SalePrice"].skew()))
print("target kurtosis: {}".format(df_train["SalePrice"].kurtosis()))

sns.displot(data=df_train, x="SalePrice")

Applying a log transformation to the target variable removes much of the kurtosis and brings it closer to the normal distribution.

The log transform also improves the model score.

## Check which features are categorical and which ones are numerical.

In [ ]:
features_numerical = list(df_train.select_dtypes(np.number))
print(f"Data set contains {len(features_numerical)} numerical features: {features_numerical}")
print()
features_categorical = list(df_train.select_dtypes("object"))
print(f"Data set contains {len(features_categorical)} categorical features: {features_categorical}")

Next, let's plot the target vs a few selected features that can be expected to have an impact on the sale price. For instance, bigger houses are more expensive ("GrLivArea") and so are houses in better condition ("OverallQual").

In [ ]:
sns.relplot(data=df_train, x="GrLivArea", y="SalePrice")

# Data wrangling

## Check for missing values

First find out how many values are missing and which features are affected by calculating the percentage of missing values for all features:

In [ ]:
missing = df_train.isnull().sum() / df_train.shape[0]
missing.sort_values(ascending=False).head(10)

There are no features with missing values *IF* the data is read with correct interpretation of "NA" values. If the default parameters of pandas.read_csv() are used, "NA" is interpreted as missing data.

## Identify and remove outliers

The scatter plot of sale price vs. ground floor area "GrLivArea" shows two cases with the biggest areas but low prices. We will drop those:

In [ ]:
df_train = df_train.loc[df_train["GrLivArea"] < 4500]

We will also drop the two homes with sales price above USD 700,000 from the training data set:

In [ ]:
if LOG_TRANSFORM_TARGET:
    outlier_price_limit = np.log(700000)
else:
    outlier_price_limit = 700000
    
df_train = df_train.loc[df_train["SalePrice"] < outlier_price_limit]

In [ ]:
print(f"The data set contains now {df_train.shape[0]} cases.")

## Split data set into training and testing set

Removing the target variable from the test data set to make sure no information leaking happens (as was the case in the notebook versions before version 7). 

In [ ]:
df_train, df_test = sklearn.model_selection.train_test_split(df_train, test_size=0.3, random_state=234)
y_train = df_train["SalePrice"]
X_train = df_train.drop(columns=["SalePrice"])
y_test = df_test["SalePrice"]
X_test = df_test.drop(columns=["SalePrice"])

# Feature engineering

Encapsulating the different steps during feature engineering in functions. The goal is to build a feature engineering pipeline that can be used to transform data sets. The encoders and transformers in the pipeline can be trained on the training data. After training, the pipeline can be used to do reproducible feature engineering on the test data set.

### Feature engineering functions

In [ ]:
def apply_encoder(X, encoder=None):
    """Applies an encoder from scikit learn to features.
    
    Args:
    X (pandas.DataFrame): features
    encoder (XXX): encoder object
    
    Returns:
    (pandas.DataFrame): features after encoder.transform(X)
    """
    return encoder.transform(X)

## Convert categorical features to numerical

Linear regression can only be used with numerical data. Therefore, we have to convert the categorical features in this data set into numerical ones.

For some of the categorical features we can guess a ranking of the different categories and use **ordinal encoding**. For instance, for the quality of the garage ("GarageQUal"), good ("Gd") sounds better than fair ("Fa") and might result in a higher sale price.

In [ ]:
def train_ordinal_encoder(X):
    """Trains an ordinal encoder on data set.
    
    Args:
    X (pandas.DataFrame): data set
    
    Returns:
    (): trained encoder object
    """
    bsmt_fin_type_to_num = {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
    bsmt_fin_type_cat = ["BsmtFinType1", "BsmtFinType2"]
    bsmt_fin_type_mapping = [{"col": col, "mapping": bsmt_fin_type_to_num} for col in bsmt_fin_type_cat]
    
    cat_qual = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", 
                "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
    cat_qual_to_num = {"NA": 0, "Po": 1, "Fa": 2, "TA": 4, "Gd": 5, "Ex": 6}
    cat_qual_mapping = [{"col": col, "mapping": cat_qual_to_num} for col in cat_qual]

    cat_paved = ["Street", "Alley", "PavedDrive"]
    cat_paved_to_num = {"NA": 0, "N": 0, "Grvl": 1, "P": 1, "Pave": 2, "Y": 2}
    cat_paved_mapping = [{"col": col, "mapping": cat_paved_to_num} for col in cat_paved]

    utilities_mapping = [{"col": "Utilities", "mapping": {"ELO": 0, "NoSeWa": 1, "NoSewr": 2, "AllPub": 3}}]
    ac_mapping = [{"col": "CentralAir", "mapping": {"N": 0, "Y": 1}}]
    garage_finish_mapping = [{"col": "GarageFinish", "mapping": {"NA": 0, "Unf": 1, "Rfn": 2, "Fin": 3}}]

    mapping_list = cat_qual_mapping + cat_paved_mapping + utilities_mapping + ac_mapping + garage_finish_mapping + bsmt_fin_type_mapping

    encoder = ce.OrdinalEncoder(mapping=mapping_list)
    encoder.fit(X)
    
    return encoder

Other categorical features are hard to rank without detailed local knowledge but can be very important. Location is a feature that has a big effect on sale prices. Therefore, it is likely that "Neighborhood" is a key feature. We will use **target mean encoding** for these features.

Target mean encoders have to be trained using the training features and targets.

In [ ]:
def train_target_encoder(X, y, cols):
    encoder = ce.TargetEncoder(cols=cols)
    #encoder = ce.LeaveOneOutEncoder(cols=cols)
    encoder.fit(X, y)
    
    return encoder

## Drop selected columns

In [ ]:
def drop_categorical_features(X):
    cat_features = list(X.select_dtypes("object"))
    X = X.drop(cat_features, axis=1)
    print(f"Dropped {len(cat_features)} columns: {cat_features}")
    return X

In [ ]:
def drop_columns(X, cols=[]):
    return X.drop(columns=cols)

## Cleaning

Some numeric features contain "NA" values, we will replace them with 0.0.

In [ ]:
def clean_NA(df, cols):
    for col in cols:
        df[col] = df[col].apply(lambda x: 0.0 if x == "NA" else float(x))
    return df

## Feature creation

There are 4 features describing the number of bathrooms. Let's replace it with a single one.

In [ ]:
def total_baths(df):
    """Consolidates information on number of baths into one feature."""
    df["Bath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    df = df.drop(columns=["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"])
    return df

Combine basement areas and their ratings. The ratings are converted into numerics by the ordinal data encoding. Combining this in one single feature makes the total basement area redundant.

In [ ]:
def finished_basement(df):
    df["FinBsmt"] = df["BsmtFinType1"] * df["BsmtFinSF1"] + df["BsmtFinType2"] * df["BsmtFinSF2"] + df["BsmtUnfSF"]
    df = df.drop(columns=["BsmtFinType1", "BsmtFinSF1", "BsmtFinType2", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF"])
    return df

Helper function to add higher-orders

In [ ]:
def add_power_2(df, col):
    df[col + "^2"] = df[col]**2
    return df

## Apply feature engineering functions

### Create feature engineering pipeline

In [ ]:
feature_eng_pipeline = []

Some of the numerical columns contain non-numerical data such as sporadic "NA" entries. Seems this concerns mostly the test data set

In [ ]:
unclean_cols = ["BsmtHalfBath", "BsmtFullBath", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "GarageArea", 
                "GarageCars", "MasVnrArea"]
feature_eng_pipeline.append(partial(clean_NA, cols=unclean_cols))

Feature encoding

In [ ]:
ordinal_encoder_obj = train_ordinal_encoder(X_train)
feature_eng_pipeline.append(partial(apply_encoder, encoder=ordinal_encoder_obj))

In [ ]:
target_encoder_cols = ["BldgType", "BsmtExposure", "Condition1", "Condition2", "Electrical", "Exterior1st", "Exterior2nd",
                       "Fence", "Foundation", "Functional", "Heating", 
                       "HouseStyle", "LandContour", "LotConfig", "LotShape", "LandSlope", "MasVnrType", "Neighborhood", "MSSubClass", "MSZoning", "RoofMatl", "RoofStyle", 
                       "SaleType", "SaleCondition"]
target_encoder_obj = train_target_encoder(X_train, y_train, target_encoder_cols)
feature_eng_pipeline.append(partial(apply_encoder, encoder=target_encoder_obj))

Feature creation

In [ ]:
feature_eng_pipeline.append(total_baths)
feature_eng_pipeline.append(finished_basement)

Add higher order polynominals to selected features

In [ ]:
feature_eng_pipeline.append(partial(add_power_2, col="OverallQual"))
feature_eng_pipeline.append(partial(add_power_2, col="KitchenQual"))
feature_eng_pipeline.append(partial(add_power_2, col="BsmtQual"))

Drop some columns
* Id: id number, has no influence on price
* GarageYrBlt: in most cases, this is the same year as the house was built. As it is highly correlated to YearBuilt, dropping for now. Later we could try to create a feature identifying homes with new garages or similar.
* LotFrontage: Linear feet of street connected to property. Contains a lot of "NA" values and is likely correlated to the size of the lot.
* MiscFeature: type of misc. feature such as "shed". As the value of the feature is in MiscVal, this probably doesn't add much extra information.
* 1stFlrSF and 2ndFlrSF: area of first and second floors. We will use the total area GrLivArea only.
* GarageCond: highly corelated with GarageQual (0.94)
* Exterior2nd: highly correlated with Exterior1st (0.92)
* PoolQC: highly correlated with PoolArea (0.91)
* SaleCondition: highly correlated with SaleType (0.91)
* GarageCars: highly correlated with GarageArea (0.90)
* FirePlaces: highly correlated with FireplaceQual (0.87) and has lower correlation to target than FireplaceQual
* TotRmsAbvGrd: highly correlated with GrLivArea (0.83) but lower correlation with target.
* ExterQual: highly correlated with OverallQual (0.72) but lower correlation with target.

In [ ]:
drop_cols = ["Id", "GarageYrBlt", "GarageType", "LotFrontage", "MiscFeature", "1stFlrSF", "2ndFlrSF"]
drop_cols += ["Fireplaces", "GarageCars", "GarageCond", "Exterior2nd", "ExterQual",
              "PoolQC", "SaleCondition", "TotRmsAbvGrd"]
feature_eng_pipeline.append(partial(drop_columns, cols=drop_cols))
#feature_eng_pipeline.append(drop_categorical_features)

Convert data to float

In [ ]:
feature_eng_pipeline.append(lambda x: x.astype(np.float64))

### Apply feature engineering pipeline to data sets

In [ ]:
X_train = reduce(lambda x, y: y(x), feature_eng_pipeline, X_train)
print(X_train.shape)
X_test = reduce(lambda x, y: y(x), feature_eng_pipeline, X_test)
print(X_test.shape)

In [ ]:
X_train.columns

## Take a look at data after feature engineering

Check which features have the highest correlation with the target:

In [ ]:
feature_target_corr = X_train.corrwith(y_train).sort_values(key=np.abs, ascending=False)
feature_target_corr.head(20)

Not surprisingly, the overall quality ("OverallQual"), living area ("GrLivArea"), neighborhood, and external quality seem to have the largest influence on the sale price.

This is closely followed by the size of garage (size in number of cars "GarageCars" and "GarageArea" are highly correlated).

The engineered feature consolidating the number of baths ("Bath") turns also out as important.

Next in line is the quality of the kitchen "KitchenQual".

The engineered feature describing the finished basement ("FinBsmt") also correlates with the sales price.

### Pairwise correlation of features

Linear regression assumes the absence of perfect multicollinearity between features. Collinearity as such does not influence the predictive power of the model. It only makes it harder to interpret how individual features influence the target. 

Nevertheless, it can be helpful to remove highly correlated features. Let's check for pairwise correlation between features:

In [ ]:
X_train_corr = X_train.corr()
x_train_corr_ranked = (X_train_corr.abs()
                       .where(np.triu(np.ones(X_train_corr.shape), k=1)
                       .astype(bool))
                       .stack()
                       .sort_values(ascending=False)
                      )
x_train_corr_ranked.head(20)

### Plot data for selected features

One key [assumption of linear regression](https://en.wikipedia.org/wiki/Linear_regression#Assumptions) is *linearity*. It is therefore helpful to take a closer look at some plots showing the dependence of the target on individual features.

Plotting the features with the highest correlation with the target:

In [ ]:
_, axes = plt.subplots(4, 4, sharey=True, figsize=(14,14))
for feat in zip(feature_target_corr.iloc[:16].index, axes.reshape(-1)):
    sns.regplot(data=pd.concat([X_train, y_train], axis=1),
               y="SalePrice",
               x=feat[0],
               ax=feat[1])

Some of the features look like the model could be improved by introducing higher orders. For instance, we added the second order of "OverallQual" to the model in the feature engineering pipeline.

# Linear Regression

## Fit regression model

In [ ]:
# selecting the best features with f_regression does not seem to improve results
#select = sklearn.feature_selection.SelectKBest(score_func=sklearn.feature_selection.f_regression, k=30)

regress = sklearn.linear_model.Ridge(alpha=10.0)

pipe = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(),
    #select,
    regress
)
pipe.fit(X_train, y_train)

regr_coef = pd.Series(regress.coef_, index=X_train.columns).sort_values(key=np.abs, ascending=False)
print(regr_coef.head(20))
#sns.lineplot(data=regr_coef)

In [ ]:
y_test_predict = pipe.predict(X_test)
y_train_predict = pipe.predict(X_train)

Reverse the log scaling of the target:

In [ ]:
if LOG_TRANSFORM_TARGET:
    y_test = np.exp(y_test)
    y_test_predict = np.exp(y_test_predict)
    y_train = np.exp(y_train)
    y_train_predict = np.exp(y_train_predict)

## Validate regression model

Calculate residuals for train and test data:

In [ ]:
def compare_true_predict(train_true, train_predict, test_true, test_predict):
    train = pd.DataFrame({"true": train_true, "predict": train_predict})
    train["label"] = "train"
    test = pd.DataFrame({"true": test_true, "predict": test_predict})
    test["label"] = "test"
    true_predict = pd.concat([train, test], axis=0)
    true_predict["residual"] = true_predict["predict"] - true_predict["true"]
    return true_predict

true_predict = compare_true_predict(y_train, y_train_predict, y_test, y_test_predict)

### Residual plot

In [ ]:
_, (ax1, ax2) = plt.subplots(1,2, figsize=(15,4))
sns.scatterplot(data=true_predict, x="true", y="residual", hue="label", ax=ax1)
sns.rugplot(data=true_predict, y="residual", hue="label", ax=ax1)
#sns.kdeplot(data=true_predict, x="residual", hue="label", common_norm=False, ax=ax2)
_ = sm.qqplot(true_predict["residual"], line="s", ax=ax2)

The residuals are roughly evenly distributed around zero for houses priced below USD 300,000. The model tends to underestimate prices of more expensive homes and the prediction error increases. This means that the [assumption of constant variance](https://en.wikipedia.org/wiki/Linear_regression#Assumptions) used for linear models is not fulfilled.

The data contains two homes priced above USD 700,000. They are much more expensive than the rest. Maybe the model would perform better if we would treat the two most expensive homes as outliers and remove them from the training data? Outliers removed in Version 14, doesn't seem to influence results.

### Test residual distribution for normality

In [ ]:
shapiro_test = stats.shapiro(true_predict["residual"])
print(f"Shapiro-Wilk statistics: {shapiro_test.statistic}, p={shapiro_test.pvalue}")
if shapiro_test.pvalue < 0.05:
    print("Residuals are not normally distributed.")
else:
    print("Residuals are normally distributed.")

The [Shapiro-Wilk test](https://en.wikipedia.org/wiki/Shapiro%E2%80%93Wilk_test) shows that the residuals are not normally distributed. For the large number of samples used here, the Shapiro-Wilk test might detect small variations from the normal distribution.

## Compute some metrics

In [ ]:
metric_r2 = (sklearn.metrics.r2_score(y_test, y_test_predict) , 
             sklearn.metrics.r2_score(y_train, y_train_predict))
print(f"R2: {metric_r2}")
metric_max_error = (sklearn.metrics.max_error(y_test, y_test_predict), 
                    sklearn.metrics.max_error(y_train, y_train_predict))
print(f"Max error: {metric_max_error}")
metric_eval = (np.sqrt(sklearn.metrics.mean_squared_error(np.log(y_test), np.log(y_test_predict))), 
               np.sqrt(sklearn.metrics.mean_squared_error(np.log(y_train), np.log(y_train_predict))))
print(f"Score: {metric_eval}")

# Submission

Load test data

In [ ]:
df_test = load_data("test.csv")
X_test = df_test
test_ids = X_test["Id"]
print(X_test.shape)

Apply feature engineering pipeline to test data

In [ ]:
X_test = reduce(lambda x, y: y(x), feature_eng_pipeline, X_test)
print(X_test.shape)

## Predict and submit

In [ ]:
y_test_predict = pipe.predict(X_test)
if LOG_TRANSFORM_TARGET:
    y_test_predict = np.exp(y_test_predict)

In [ ]:
submission = pd.DataFrame({"Id": test_ids, "SalePrice": y_test_predict})
submission.head()

In [ ]:
def write_data(df, filename):
    """Writes data to file.
    
    Args:
    df (pandas.DataFrame): data
    filename (str): filename
    """
    
    data_dir = "/kaggle/working"
    data_file = os.path.join(data_dir, filename)
    df.to_csv(data_file, index=False)

write_data(submission, "submission.csv")

# License

Copyright (c) 2022 Andreas Klust

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Changelog

## Version 15 - score 0.139
- Improved documentation
- Add some 2nd order polynominals to selected features

## Version 14 - score 0.138
- Remove most expensive homes from training set
- Drop features with highest cross-correlation (above 0.7)

## Version 13 - score 0.136
- Add residual plot
- Limiting number of dropped features

## Version 12 - score 0.137
- Some improvements with feature engineering

## Version 10
- Switch to seaborn for visualization

## Version 9 - score 0.139
- Fix submission file format

## Version 8
- Prepare competition submission
- Fix score calculation, now using root mean square instead of mean square

## Version 7
- Create feature engineering pipeline to make feature engineering reproducible
- Stop leaking target information when doing target mean encoding of test features

## Version 6
- Apply log transformation to target

## Version 5
- Residual plot
- Feature engineering: higher order of OverallQual and OverallCond lead to improvement
- Feature engineering: consolidate bathrooms

## Version 4
- Linear regression

## Version 3
- Convert categorical features to numeric

## Version 2
- Remove some outliers

## Version 1
- Read training data and create some basic plots
- Check for missing values